# Realistic KMeans clustering example - using Penguins

## Import Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

## Read data

In [ ]:
penguins = sns.load_dataset('penguins')

In [ ]:
penguins.info()

## Preprocess

We know we need to standardize to REMOVE the magnitude and scale dominated by 1 variable.

In [ ]:
sns.catplot(data = penguins, kind='box', aspect=2)

plt.show()

Extract the numeric columns.

In [ ]:
pens_features = penguins.select_dtypes('number').copy()

Standardize using the `StandardScaler()` method.

In [ ]:
from sklearn.preprocessing import StandardScaler

INITIALIZE, FIT, and TRANSFORM in 1 line of code!!!

In [ ]:
Xpens = StandardScaler().fit_transform( pens_features )

In [ ]:
Xpens.shape

In [ ]:
type( Xpens )

In [ ]:
pens_features.shape

In [ ]:
sns.catplot(data = pd.DataFrame(Xpens, columns=pens_features.columns), kind='box', aspect=2)

plt.show()

## KMeans

We know there are 3 species in the `penguins` data set.

In [ ]:
penguins.species.value_counts()

However, let's follow GOOD practice and start out by applying KMeans with 2 clusters!!!!!!

In [ ]:
from sklearn.cluster import KMeans

INITIALIZE, FIT, and PREDICT in one line of code!!!!!

In [ ]:
clusters_2 = KMeans(n_clusters=2, random_state=121, n_init=25, max_iter=500).fit_predict( Xpens )

The reason why it is ALWAYS critical to FOLLOW the basic steps for EDA is we NEED TO always REMEMBER if our data set has MISSINGS!!!

In [ ]:
penguins.info()

In [ ]:
penguins.isna().sum()

In [ ]:
penguins.head(10)

## Drop missings

The `StandardScaler()` method **DROPS MISSINGS** behind the scenes!!!!

The `KMeans()` methods **CANNOT HANDLE** missings!!!!

But the SIMPLEST and MOST BASIC action for dealing with MISSINGS is to REMOVE THEM!!!

REMOVING any row with at least 1 missing will return the **COMPLETE CASES**!!!!!

In [ ]:
penguins_clean = penguins.dropna().copy()

In [ ]:
penguins_clean.info()

In [ ]:
penguins.shape

In [ ]:
penguins_clean.shape

In [ ]:
penguins_clean.isna().sum()

We MUST now PREPROCESS the CLEANED dataframe and repeat the steps!

In [ ]:
pens_features_clean = penguins_clean.select_dtypes('number').copy()

STANDARDIZED the cleaned numeric columns

In [ ]:
X = StandardScaler().fit_transform( pens_features_clean )

In [ ]:
X.shape

In [ ]:
pens_features_clean.shape

In [ ]:
sns.catplot(data = pd.DataFrame(X, columns=pens_features_clean.columns), kind='box', aspect=2)

plt.show()

Now we can execute KMeans clustering!!!

In [ ]:
clusters_2 = KMeans(n_clusters=2, random_state=121, n_init=25, max_iter=500).fit_predict( X )

Assign the cluster labels to columns in a COPY of the cleaned data.

In [ ]:
penguins_clean_copy = penguins_clean.copy()

In [ ]:
penguins_clean_copy['k2'] = pd.Series( clusters_2, index=penguins_clean_copy.index ).astype('category')

In [ ]:
penguins_clean_copy.info()

In [ ]:
penguins_clean_copy.k2.value_counts()

In [ ]:
sns.pairplot(data = penguins_clean_copy, hue='k2', diag_kws={'common_norm': False})

plt.show()

### 3 clusters

Since we know there are 3 species...let's now create 3 clusters and then compare the clusters with the KNOWN groupings!

In [ ]:
clusters_3 = KMeans(n_clusters=3, random_state=121, n_init=25, max_iter=500).fit_predict( X )

In [ ]:
penguins_clean_copy['k3'] = pd.Series( clusters_3, index=penguins_clean_copy.index ).astype('category')

In [ ]:
penguins_clean_copy.k3.value_counts()

In [ ]:
sns.pairplot(data = penguins_clean_copy, hue='k3', diag_kws={'common_norm': False})

plt.show()

Use a heatmap to visually show how well the CLUSTERS  align with the KNOWN groupings!

In [ ]:
fig, ax = plt.subplots()

sns.heatmap(data = pd.crosstab( penguins_clean_copy.species, penguins_clean_copy.k3, margins=True ), 
            annot=True, annot_kws={"fontsize": 20}, fmt='g',
            cbar=False,
            ax=ax)

plt.show()

Look at the scatter plot between 2 of the variables. Color by `k3` and set the marker shape by `species`.

In [ ]:
sns.relplot(data = penguins_clean_copy, x='bill_length_mm', y='bill_depth_mm', hue='k3', style='species')

plt.show()

## Optimal number of clusters

We need the KNEE BEND PLOT!

In [ ]:
tots_within = []

K = range(1, 31)

for k in K:
    km = KMeans(n_clusters=k, random_state=121, n_init=25, max_iter=500)
    km = km.fit( X )
    
    tots_within.append( km.inertia_ )

Visualize the KNEE BEND plot.

In [ ]:
fig, ax = plt.subplots()

ax.plot( K, tots_within, 'bo-' )
ax.set_xlabel('number of clusters')
ax.set_ylabel('total within sum of squares')

plt.show()

What if we would use 5 clusters based on the KNEE BEND?

In [ ]:
clusters_5 = KMeans(n_clusters=5, random_state=121, n_init=25, max_iter=500).fit_predict( X )

In [ ]:
penguins_clean_copy['k5'] = pd.Series( clusters_5, index=penguins_clean_copy.index ).astype('category')

In [ ]:
sns.pairplot(data = penguins_clean_copy, hue='k5', diag_kws={'common_norm': False})

plt.show()